# Corporacion Favorita - New Superb Forecasting Model - 

## Split and Model Pipeline

#codi

Made by 4B Consultancy (Janne Heuvelmans, Georgi Duev, Alexander Engelage, Sebastiaan de Bruin) - 2024

In this data pipeline, 

The following steps are made within this notebook:  

>-0. Import Packages 

>-1. Load final dataset and aggregate dataset to weekly level
    -1.1 Load final dataset made in Data Preperation Pipeline Notebook
    -1.2 Aggregate dataset to weekly level

>-2. Column transformers and Train, Test, Validation Split

>-3. Models

>-4. Pick best model one and optimize with grid search

## 0. Import Packages

In [1]:
# Importing the libraries
import pandas as pd
import numpy as np
import polars as pl
import os
import sys
import altair as alt
import vegafusion as vf
import sklearn
import time
from datetime import date, datetime, timedelta
from sklearn.pipeline import Pipeline, make_pipeline

In [2]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

from sklearn.metrics import mean_absolute_percentage_error

import statsmodels.api as sm

In [3]:
from sklearn.model_selection import train_test_split

## 1. Load final dataset, Inpute Stockouts and Aggregate dataset to weekly level

### 1.1. Functions - Import raw data from local PATH
Create import data function and give basic information function within the importing function.

Return basic information on each dataframe:  
- a) Information on the number of observation and features.  
- b) Information on the size of the dataframe. 

TO-DO: Import via polars, and use polars dataframe?

In [4]:
def f_get_data_and_info(import_path, file_name="df_final"):

    print(f"\nReading file {file_name}\n")

    # Load data.
    df = pd.read_parquet(import_path + file_name + ".parquet")

    # Getting the basic information of the dataframe (number of observations and features, and size)
    print(
        f"The '{file_name}' dataframe contains: {df.shape[0]:,}".replace(",", ".")
        + f" observations and {df.shape[1]} features."
    )
    print(
        f"Prepared and transformed dataframe has optimized size of {round(sys.getsizeof(df)/1024/1024/1024, 2)} GB."
    )

    return df

### 1.2. Importing raw data
Importing parquet files with importing function (giving basic information)

In [5]:
import_path = "C:/Users/alexander/Documents/0. Data Science and AI for Experts/TEST/"


# Importing final df
df_final = f_get_data_and_info(import_path, file_name="Prepped_data_20241004")


Reading file Prepped_data_20241004

The 'Prepped_data_20241004' dataframe contains: 67.834.270 observations and 19 features.
Prepared and transformed dataframe has optimized size of 2.72 GB.


To-do: include null_count print dunction into importing OR make basic descrption function with features, size, null_count

In [6]:
df_final.info()
# Count nulls per column
null_counts = df_final.isnull().sum()

# Print results
for column, count in null_counts.items():
    print(f"Column '{column}' has {count} null values.")

<class 'pandas.core.frame.DataFrame'>
Index: 67834270 entries, 0 to 67834269
Data columns (total 19 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   store_nbr               uint8         
 1   item_nbr                int32         
 2   date                    datetime64[ns]
 3   unit_sales              float32       
 4   onpromotion             bool          
 5   holiday_local_count     int8          
 6   holiday_national_count  int8          
 7   holiday_regional_count  int8          
 8   store_type              category      
 9   store_cluster           uint8         
 10  item_family             category      
 11  item_class              uint16        
 12  perishable              uint8         
 13  store_status            int8          
 14  item_status             int8          
 15  year                    int16         
 16  weekday                 int8          
 17  week_nbr                int8          
 18  week_

## 2.0 Train test val split

SKtime

ExpandingWindowSplitter


#TO-DO: Selecting on weeks or via data?

In [7]:
features = [
    "store_nbr",
    "item_nbr",
    "onpromotion",
    "holiday_local_count",
    "holiday_national_count",
    "holiday_regional_count",
    "store_type",
    "store_cluster",
    "item_family",
    "item_class",
    "perishable",
    "store_status",
    "item_status",
    "year",
    # "week_nbr",
    "week_number_cum",
]

target_variable = ["unit_sales"]

In [8]:
def train_val_test_split(df, features, target_variable, window_length=26):

    # Sort the DataFrame by store number, item number, and date for ordering
    df = df.sort_values(["store_nbr", "item_nbr", "week_number_cum"])

    # Create X (features) and y (target)
    X = df[features]
    y = df[target_variable]

    # Get the maximum week in the dataset
    max_week = df["week_number_cum"].max()

    # Calculate start and end weeks for validation and test sets
    test_week_start = max_week - window_length + 1

    val_week_start = max_week - 2 * window_length + 1

    val_week_end = test_week_start - 1

    train_week_end = val_week_start - 1

    # Train data: All data before the start of the validation period
    X_train = X[X["week_number_cum"] <= train_week_end]
    y_train = y[df["week_number_cum"] <= train_week_end]

    # Val data: From `val_week_start` to `val_week_end`
    X_val = X[
        (X["week_number_cum"] >= val_week_start)
        & (X["week_number_cum"] <= val_week_end)
    ]
    y_val = y[
        (df["week_number_cum"] >= val_week_start)
        & (df["week_number_cum"] <= val_week_end)
    ]

    # Test data: From `test_week_start` to `max_week`
    X_test = X[
        (X["week_number_cum"] >= test_week_start) & (X["week_number_cum"] <= max_week)
    ]
    y_test = y[
        (df["week_number_cum"] >= test_week_start) & (df["week_number_cum"] <= max_week)
    ]

    # Function to print split information
    def print_split_info(split_name, X_split, y_split):
        print(f"\n{split_name} set:")
        print(f"X_{split_name.lower()} shape: {X_split.shape}")
        print(f"y_{split_name.lower()} shape: {y_split.shape}")
        print(f"{split_name} Min Week: {X_split['week_number_cum'].min()}")
        print(f"{split_name} Max Week: {X_split['week_number_cum'].max()}")
        print(f"{split_name} number of weeks: {X_split['week_number_cum'].nunique()}")
        print(f"Number of stores: {X_split['store_nbr'].nunique()}")
        print(f"Number of items: {X_split['item_nbr'].nunique()}")

    # Print information about the splits
    print_split_info("Train", X_train, y_train)
    print_split_info("Validation", X_val, y_val)
    print_split_info("Test", X_test, y_test)

    return X_train, y_train, X_val, y_val, X_test, y_test

In [9]:
X_train, y_train, X_val, y_val, X_test, y_test = train_val_test_split(
    df_final, features, target_variable, window_length=26
)

KeyboardInterrupt: 

train_val_test_split without creating X (features) and y (target)

In [10]:
def train_val_test_split(df, window_length=26):

    # Sort the DataFrame by store number, item number, and date for ordering
    df = df.sort_values(["store_nbr", "item_nbr", "week_number_cum"])

    # Get the maximum week in the dataset
    max_week = df["week_number_cum"].max()

    # Calculate start and end weeks for validation and test sets
    test_week_start = max_week - window_length + 1

    val_week_start = max_week - 2 * window_length + 1

    val_week_end = test_week_start - 1

    train_week_end = val_week_start - 1

    # Train data: All data before the start of the validation period
    train = df[df["week_number_cum"] <= train_week_end]

    # Val data: From `val_week_start` to `val_week_end`
    val = df[
        (df["week_number_cum"] >= val_week_start)
        & (df["week_number_cum"] <= val_week_end)
    ]

    # Test data: From `test_week_start` to `max_week`
    test = df[
        (df["week_number_cum"] >= test_week_start) & (df["week_number_cum"] <= max_week)
    ]

    # Function to print split information
    def print_split_info(split_name, split):
        print(f"\n{split_name} set: shape: {split.shape}")
        print(f"{split_name} Min Week: {split['week_number_cum'].min()}")
        print(f"{split_name} Max Week: {split['week_number_cum'].max()}")
        print(f"{split_name} number of weeks: {split['week_number_cum'].nunique()}")
        print(f"Number of stores: {split['store_nbr'].nunique()}")
        print(f"Number of items: {split['item_nbr'].nunique()}")

    # Print information about the splits
    print_split_info("Train", train)
    print_split_info("Validation", val)
    print_split_info("Test", test)

    return train, val, test

In [11]:
train, val, test = train_val_test_split(df_final, window_length=26)


Train set: shape: (53398880, 19)
Train Min Week: 1
Train Max Week: 190
Train number of weeks: 190
Number of stores: 10
Number of items: 4021

Validation set: shape: (7318220, 19)
Validation Min Week: 191
Validation Max Week: 216
Validation number of weeks: 26
Number of stores: 10
Number of items: 4021

Test set: shape: (7117170, 19)
Test Min Week: 217
Test Max Week: 242
Test number of weeks: 26
Number of stores: 10
Number of items: 4021


To-do: do we split based on dates or based on weeks since start?

In [10]:
# X_train, y_train, X_test, y_test, X_val, y_val = train_test_val_split(
#     df_final, train_end="2016-06-01", test_end="2017-01-01"
# )

## 3.0 Functions - Impute stockouts and Aggregate dataset to weekly level


#### 3.1. Impute stockouts

Stockout on store level

•      Perishable good: when there are missing values for two consecutive days for a given item per individual store 

•      Nonperishable goods: when there are missing values for 7 consecutive days for a given item and per individual store

•      Action: Impute with Rolling Mean with defeault window of 7 days 

------------------------------------

In [12]:
def impute_stockouts_polars(df_pandas, window_size=7):

    # Convert the input Pandas DataFrame to a Polars DataFrame
    df = pl.from_pandas(df_pandas)

    # Sort the DataFrame by store number, item number, and date for ordering

    df = df.sort(["store_nbr", "item_nbr", "date"])

    # Nested function calc_missing_count to calculate the count of consecutive missing values in unit_sales

    def calc_missing_count(unit_sales):

        return (
            unit_sales.is_null()  # Check for null values
            .cast(pl.Int32)  # Cast to integer (1 for null, 0 for not null)
            .cum_sum()  # Cumulative sum to count sequential nulls
            .over(["store_nbr", "item_nbr"])  # Group by store_nbr and item_nbr
        )

    # Nested function to Inpute with rolling mean for missing values
    def rolling_mean_imputation(unit_sales, window_size):

        return (
            unit_sales.rolling_mean(
                window_size=window_size, min_periods=1
            )  # Impute strategy based on rolling mean
            .shift(
                1
            )  # Shift window by one day, to prevent taking the same day into account
            .over(["store_nbr", "item_nbr"])  # Group by store_nbr and item_nbr
        )

    # Apply the imputation logic based on the perishable status of the items

    df = df.with_columns(
        [
            pl.when(pl.col("perishable") == 1)  # Check if the item is perishable = 1
            .then(
                pl.when(
                    calc_missing_count(pl.col("unit_sales")) == 1
                )  # 1 missing value
                .then(0)  # --> Impute with 0
                .when(
                    calc_missing_count(pl.col("unit_sales")) > 2
                )  # More than 2 missing values
                .then(0)  # --> Impute with 0
                .when(
                    calc_missing_count(pl.col("unit_sales")) == 2
                )  # = 2 missing values
                .then(
                    rolling_mean_imputation(pl.col("unit_sales"), window_size)
                )  # --> Inpute with rolling mean for 2 missing days
                .otherwise(pl.col("unit_sales"))  # Otherwise keep original value
            )
            .when(pl.col("perishable") == 0)  # If the item is not perishable = 0
            .then(
                pl.when(
                    calc_missing_count(pl.col("unit_sales")) > 7
                )  # More than 7 missing values
                .then(0)  # --> Impute with 0
                .when(
                    calc_missing_count(pl.col("unit_sales")) <= 7
                )  # if less 7 missing values
                .then(
                    rolling_mean_imputation(pl.col("unit_sales"), window_size)
                )  # --> Inpute with rolling mean for missing 7 or less days
                .otherwise(pl.col("unit_sales"))  # Otherwise keep original value
            )
            .otherwise(pl.col("unit_sales"))  # For any other case not covered
            .alias("unit_sales")  # Alias the new column as 'unit_sales'
        ]
    )

    # Convert Polars df back to Pandas df
    df = df.to_pandas()

    return df

In [12]:
# def impute_stockouts_(df):

#     df = impute_stockouts_polars(df, window_size=7)

#     return df

## 3.2. Aggregate dataset to weekly level

- Group the DataFrame by store number, item number, year, and week_cum_number, then aggregate the columns
--> "unit_sales","onpromotion", "holiday_local_count","holiday_regional_count","holiday_national_count",


In [13]:
def aggregate_week(df):

    # Sort the DataFrame by store number, item number, and date for ordering
    df = df.sort_values(["store_nbr", "item_nbr", "year", "week_nbr"])

    # Group by the specified columns and aggregate
    df = (
        df.groupby(
            [
                "store_nbr",
                "item_nbr",
                "year",
                "week_number_cum",  # Aggregating by week_number_cum
            ]
        )
        .agg(
            {
                "unit_sales": "sum",
                "onpromotion": "sum",
                "holiday_local_count": "sum",
                "holiday_regional_count": "sum",
                "holiday_national_count": "sum",
                "date": "first",  # Keep the first day of week, needed to run Timeseries models from SKtime
                "store_type": "first",  # Keep the first occurrence of store_type
                "store_cluster": "first",  # Keep the first occurrence of store_cluster
                "item_family": "first",  # Keep the first occurrence of item_family
                "item_class": "first",  # Keep the first occurrence of item_class
                "perishable": "first",  # Keep the first occurrence of perishable
                "store_status": "last",  # Keep the last occurrence of store_status
                "item_status": "last",  # Keep the last occurrence of item_status
            }
        )
        .reset_index()
    )

    return df

In [14]:
# df_agg = aggregate_week(df_final)

## 4. Column transformers

### 4.1. Column transformers

In [14]:
features = [
    "date",
    "store_nbr",
    "item_nbr",
    "onpromotion",
    "holiday_local_count",
    "holiday_national_count",
    "holiday_regional_count",
    "store_type",
    "store_cluster",
    "item_family",
    "item_class",
    "perishable",
    "store_status",
    "item_status",
    "year",
    "week_number_cum",
]

target_variable = ["unit_sales"]

In [27]:
X = df_final[features]
y = df_final[target_variable]

In [ ]:
DEF STOP

To-do: Do we need onehotencoder? --> then needed to seperate between timeseries en ML models

df_X_OneHotEncoder = pd.DataFrame(
    data    = m_neighborhoods,
    columns = OneHotEncoder().categories_[0]
).astype('int')

To-do: Change catagory dtypes from store_type and item_family just to numbers in prep pipeline?

To-do: Normalization (Min-Max scaling) or Standardization (mean of unit sd) --> Scaler needed?

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import FunctionTransformer

# X = df[features]
# y = df[target_variable]

# Create transformers with numeric and catagorical dtype features
num_features = X.select_dtypes(exclude="object").columns
cat_features = X.select_dtypes(include="object").columns

# Create a ColumnTransformer
preprocessor_pipeline = ColumnTransformer(
    transformers=[
        (
            "ImputeStockouts",  # Impute stockouts
            FunctionTransformer(impute_stockouts_polars, kw_args={"window_size": 7}),
            num_features.tolist()
            + cat_features.tolist(),  # Need to combine columns for the function
        ),
        (
            "AggregateWeekly",  # Aggregate dataset to weekly level
            FunctionTransformer(aggregate_week),
            num_features.tolist()
            + cat_features.tolist(),  # Need to combine columns for the function
        ),
        # ("OneHotEncoder", OneHotEncoder(handle_unknown='ignore'), cat_features),
        # ("Scaler", StandardScaler() or MinMaxScaler(), num_features),
    ],
    remainder="drop",
    verbose=True,
)

In [26]:
# X = df[features]
# y = df[target_variable]

# Create transformers with numeric and catagorical dtype features
num_features = X.select_dtypes(exclude="object").columns
cat_features = X.select_dtypes(include="object").columns

NameError: name 'X' is not defined

In [28]:
# Create transformers with numeric and catagorical dtype features
num_features = X.select_dtypes(exclude="object").columns
cat_features = X.select_dtypes(include="object").columns

preprocessor = ColumnTransformer(
    transformers=[
        ("onehot", OneHotEncoder(handle_unknown="ignore"), cat_features),
        ("scaler", StandardScaler(), num_features),
    ],
    remainder="passthrough",
)

NameError: name 'cat_features' is not defined

In [ ]:
pipeline_xgb = Pipeline(
    [
        ("preprocessing", preprocessor_pipeline),
        (
            "model",
            make_reduction(XGBRegressor(), window_length=13, strategy="recursive"),
        ),
    ]
)

In [15]:
def impute_agg_preprocessing(df, window_size=7):

    df = impute_stockouts_polars(df, window_size)

    df = aggregate_week(df_final)

    return df

preprocess each train, test, val split separately.
Before seperating X and y
to:do: .transform(X) or fit_transform(X)

In [38]:
def preprocess_split_fit_transform(df, features, target_variable):
    X = df[features]
    y = df[target_variable]

    # Create transformers with numeric and catagorical dtype features
    num_features = X.select_dtypes(exclude="object").columns
    cat_features = X.select_dtypes(include="object").columns

    pipeline_preprocessor = ColumnTransformer(
        transformers=[
            ("onehot", OneHotEncoder(handle_unknown="ignore"), cat_features),
            ("scaler", StandardScaler(), num_features),
        ],
        remainder="passthrough",
    )

    X_transformed = pipeline_preprocessor.fit_transform(X)

    return X_transformed, y

Splitting and preprocessing with inputation and aggergation to weekly data

In [17]:
train_df, val_df, test_df = train_val_test_split(df_final)
train_df = impute_agg_preprocessing(train_df)
val_df = impute_agg_preprocessing(val_df)
test_df = impute_agg_preprocessing(test_df)


Train set: shape: (53398880, 19)
Train Min Week: 1
Train Max Week: 190
Train number of weeks: 190
Number of stores: 10
Number of items: 4021

Validation set: shape: (7318220, 19)
Validation Min Week: 191
Validation Max Week: 216
Validation number of weeks: 26
Number of stores: 10
Number of items: 4021

Test set: shape: (7117170, 19)
Test Min Week: 217
Test Max Week: 242
Test number of weeks: 26
Number of stores: 10
Number of items: 4021


In [18]:
def null_count(df):  # Count nulls per column
    null_counts = df.isnull().sum()

    # Print results
    for column, count in null_counts.items():
        print(f"Column '{column}' has {count} null values.")

    return

In [22]:
null_count(test_df)
test_df.info()

Column 'store_nbr' has 0 null values.
Column 'item_nbr' has 0 null values.
Column 'year' has 0 null values.
Column 'week_number_cum' has 0 null values.
Column 'unit_sales' has 0 null values.
Column 'onpromotion' has 0 null values.
Column 'holiday_local_count' has 0 null values.
Column 'holiday_regional_count' has 0 null values.
Column 'holiday_national_count' has 0 null values.
Column 'date' has 0 null values.
Column 'store_type' has 0 null values.
Column 'store_cluster' has 0 null values.
Column 'item_family' has 0 null values.
Column 'item_class' has 0 null values.
Column 'perishable' has 0 null values.
Column 'store_status' has 0 null values.
Column 'item_status' has 0 null values.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9730820 entries, 0 to 9730819
Data columns (total 17 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   store_nbr               uint8         
 1   item_nbr                int32         
 2   year        

Process each split using preprocessor for X and y split
To-do: think do we need one-hot and sclaer?

In [39]:
X_train, y_train = preprocess_split_fit_transform(train_df, features, target_variable)
X_val, y_val = preprocess_split_fit_transform(val_df, features, target_variable)
X_test, y_test = preprocess_split_fit_transform(test_df, features, target_variable)

TypeError: float() argument must be a string or a real number, not 'Timestamp'

## 5. Models

### 5.1. Models list to compare in model

In [ ]:
# <PATH>.\venv_case_project\Scripts\activate

# source venv_macbook/bin/activate

# pip install sktime
# pip install statsmodels
# pip install xgboost

In [17]:
from sktime.forecasting.exp_smoothing import ExponentialSmoothing
from sktime.forecasting.compose import make_reduction
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

models = {
    "Holt-Winters": ExponentialSmoothing(trend="add", seasonal="add", sp=52),
    "Random Forest Regressor": make_reduction(
        RandomForestRegressor(), window_length=13, strategy="recursive"
    ),
    "XGBoost": make_reduction(
        XGBRegressor(tree_method="hist"), window_length=13, strategy="recursive"
    ),
}

### 5.2 Run Model

In [19]:
XGBoost.__dict__

NameError: name 'XGBoost' is not defined

In [ ]:
def train_model(X_train, y_train, window_length=13):
    model = XGBRegressor(tree_method="hist", verbosity=1)  # Memory-efficient settings
    model.fit(X_train, y_train)
    return model

In [ ]:
# Initialize a dictionary to store results
results = {}

for model_name, model in models.items():
    # Fit model
    model.fit(X_train, y_train)

    # Predict on test set
    y_pred = model.predict(X_test)

    # Calculate evaluation metrics
    mape, accuracy, bias = calculate_metrics(y_test, y_pred)

    # Store results
    results[model_name] = {
        "MAPE": mape,
        "Accuracy": accuracy,
        "Bias": bias,
    }

### 5.3. Evaulation Metrics and Evaluate Model functions

In [20]:
def calculate_metrics(y_true, y_pred):
    mape = mean_absolute_percentage_error(y_true, y_pred)
    accuracy = 1 - mape
    bias = np.mean(y_pred - y_true)
    return {"MAPE": mape, "Accuracy": accuracy, "Bias": bias}

In [ ]:
# Step 5: Model Evaluation
def evaluate_model(model, X, y, set_name="Validation"):
    y_pred = model.predict(X)
    mape = mean_absolute_percentage_error(y, y_pred)
    accuracy = 100 - mape
    bias = np.mean(y_pred - y)
    print(f"{set_name} Metrics: MAPE: {mape}, Accuracy: {accuracy}, Bias: {bias}")
    return {"MAPE": mape, "Accuracy": accuracy, "Bias": bias}

### X.4 Evaulation Metrics

In [ ]:
# Define MAPE function
def mean_absolute_percentage_error(y_true, y_pred):
    """Calculate Mean Absolute Percentage Error (MAPE)."""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    nonzero_mask = y_true != 0  # Avoid division by zero
    return (
        np.mean(
            np.abs((y_true[nonzero_mask] - y_pred[nonzero_mask]) / y_true[nonzero_mask])
        )
        * 100
    )

-------------------------------------------

In [ ]:
def run_pipeline(df, target_column="unit_sales", window_length=26):
    # Split data
    train_df, val_df, test_df = train_val_test_split(df, window_length=window_length)

    # Apply preprocessing
    train_df = custom_preprocessing(train_df)
    val_df = custom_preprocessing(val_df)
    test_df = custom_preprocessing(test_df)

    # Separate features and target
    X_train, y_train = split_features_target(train_df, target_column)
    X_val, y_val = split_features_target(val_df, target_column)
    X_test, y_test = split_features_target(test_df, target_column)

    # Train the model
    model = train_model(X_train, y_train)

    # Evaluate on validation set
    val_metrics = evaluate_model(model, X_val, y_val, set_name="Validation")

    # Evaluate on test set
    test_metrics = evaluate_model(model, X_test, y_test, set_name="Test")

    return model, val_metrics, test_metrics


# Run the entire pipeline
model, val_metrics, test_metrics = run_pipeline(df_final)

### X.4 Evaulation Metrics

Run / Fit model

Test Claude

## X. Pick best one --> Optimize with grid search

In [ ]:
# 	Get	feature	importances	from	the	model
feature_importances = best_model.get_feature_importance(prettified=False)

# 	Get	feature	names	(considering	potential	transformation)
feature_names = preprocessor.get_feature_names_out()  # 	After	column	transformation

# 	Sort	feature	importances	and	names	together	by	importance	(descending)
sorted_idx = np.argsort(feature_importances)
feature_importances = feature_importances[sorted_idx]
feature_names = feature_names[sorted_idx]

# 	Define	plot	size	and	create	a	bar	chart
plt.figure(figsize=(12, 6))
plt.barh(range(len(feature_names)), feature_importances, align="center")
plt.yticks(range(len(feature_names)), feature_names)
plt.xlabel("Feature	Importance")
plt.ylabel("Feature	Names")
plt.title("Feature	Importance	for	Electricity	Demand-Supply	Prediction")
plt.grid(axis="x", linestyle="--", alpha=0.6)
plt.show()

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "depth": [4, 6, 8],
    "learning_rate": [0.05, 0.1, 0.2],
    "iterations": [50, 100, 200],
}

best_model = CatBoostRegressor()

grid_search = GridSearchCV(estimator=best_model, param_grid=param_grid, cv=5)
grid_search.fit(X_train, y_train)
best_params = grid_search.best_params_

In [ ]:
best_param

In [ ]:
param_grid = {
    "svm__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "svm__gamma": [0.001, 0.01, 0.1, 1, 10, 100],
}
pipe = pipeline.Pipeline([("scaler", MinMaxScaler()), ("svm", SVC(C=100))])
grid = GridSearchCV(pipe, param_grid=param_grid, cv=5)
grid.fit(X_train, y_train)

In [ ]:
import multiprocessing

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import GridSearchCV

import xgboost as xgb

if __name__ == "__main__":
    print("Parallel Parameter optimization")
    X, y = fetch_california_housing(return_X_y=True)
    # Make sure the number of threads is balanced.
    xgb_model = xgb.XGBRegressor(
        n_jobs=multiprocessing.cpu_count() // 2, tree_method="hist"
    )
    clf = GridSearchCV(
        xgb_model,
        {"max_depth": [2, 4, 6], "n_estimators": [50, 100, 200]},
        verbose=1,
        n_jobs=2,
    )
    clf.fit(X, y)
    print(clf.best_score_)
    print(clf.best_params_)

In [ ]:
%%time
search.fit(X, y)

In [ ]:
# get best parameters and train full XGBoost with larger number of estimators
xgb_best_params = {
    k.split("__")[-1]: v
    for k, v in search.best_params_.items()
    if k != "xgb__n_estimators"
}
xgb_best = Pipeline(
    steps=[
        ("pre", prepare_nonlinear),
        ("xgb", XGBRegressor(n_estimators=1000, **xgb_best_params)),
    ]
)
xgb_best_params

In [ ]:
%%time
xgb_cv = cross_validate(
    xgb_best,
    X,
    y,
    scoring=["neg_mean_squared_error"],
    cv=10,
    return_train_score=True,
    return_estimator=True,
)

Nested cross-validation
https://ml-course.github.io/master/notebooks/Tutorial%203%20-%20Machine%20Learning%20in%20Python.html#evaluate

In [ ]:
scores = cross_val_score(
    GridSearchCV(SVC(), param_grid, cv=5), iris.data, iris.target, cv=5
)

In [ ]:
scores = cross_val_score(
    GridSearchCV(SVC(), param_grid, cv=5), iris.data, iris.target, cv=5
)
print("Cross-validation scores: ", scores)
print("Mean cross-validation score: ", scores.mean())

In [ ]:
df_eda.to_csv("final_model.csv", index=False)
# 	Save	the	trained	model
lr_model.save_model("catboost_model.cbm")

To-do: Residual analysis?
--> Check if errors are randomly distributed in pointcloud